# Análisis Censal Completo — Argentina 2022

Indicadores demográficos, educativos y de déficit habitacional.
Cada sección presenta: tabla con N y %, gráfico de barras, apertura provincial y por género.

> **Nota sobre apertura por género:** Los datos están en formato largo pre-agregado por radio censal. Sexo (`PERSONA_P02`) y otras variables como edad o educación son agregaciones **independientes** — no existe una variable cruzada directa en la base. Por eso las distribuciones por género se muestran en paralelo a las otras variables, no combinadas.

---
## 0. Setup

In [ ]:
import ciut
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

MUJER  = "#e07b8a"
VARON  = "#5b8ec4"
TOTAL  = "#4a4a6a"
VERDE  = "#69b3a2"
NARANJA = "#e8965a"

def fmt_n(n): return f"{n:,.0f}"
def fmt_pct(p): return f"{p:.1f}%"

def tabla_n_pct(serie, total=None):
    """Devuelve DataFrame con N y % a partir de una serie agrupada."""
    df = serie.rename("N").reset_index()
    t = total if total else serie.sum()
    df["%"] = (df["N"] / t * 100).round(1)
    return df

print("OK")

---
## 1. Descarga de datos

⏱️ Las consultas sin filtro de provincia tardan entre 10 y 40 segundos cada una.

In [ ]:
# Variables de personas
df_sexo  = ciut.query(variables="PERSONA_P02")      # Sexo
df_edad  = ciut.query(variables="PERSONA_EDADQUI")   # Grupos quinquenales
df_educ  = ciut.query(variables="PERSONA_MNI")       # Nivel de instrucción

In [ ]:
# Variables de hogares
df_nbi_tot = ciut.query(variables="HOGAR_NBI_TOT")   # NBI total
df_nbi_esc = ciut.query(variables="HOGAR_NBI_ESC")   # NBI escolaridad
df_nbi_hac = ciut.query(variables="HOGAR_NBI_HAC")   # NBI hacinamiento
df_nbi_san = ciut.query(variables="HOGAR_NBI_SAN")   # NBI saneamiento
df_nbi_sub = ciut.query(variables="HOGAR_NBI_SUB")   # NBI subsistencia
df_nbi_viv = ciut.query(variables="HOGAR_NBI_VIV")   # NBI vivienda
df_ipmh    = ciut.query(variables="HOGAR_IPMH")      # Índice privación material
df_hacin   = ciut.query(variables="HOGAR_H20CP")     # Hacinamiento

In [ ]:
# Constantes útiles
POB_TOTAL   = df_sexo["conteo"].sum()
HOGAR_TOTAL = df_nbi_tot["conteo"].sum()

# Categorías reales de sexo
cats_sexo = df_sexo.groupby("etiqueta_categoria")["conteo"].sum().sort_values(ascending=False)
CAT_MUJER = cats_sexo.index[0]   # "Mujer / Femenino"
CAT_VARON = cats_sexo.index[1]   # "Varón / Masculino"

print(f"Población total : {POB_TOTAL:,}")
print(f"Hogares totales : {HOGAR_TOTAL:,}")
print(f"Cat. mujer      : {CAT_MUJER}")
print(f"Cat. varón      : {CAT_VARON}")

---
## 2. Población por provincia — apertura por género

In [ ]:
# Tabla N y % por provincia × sexo
sexo_prov = (
    df_sexo
    .groupby(["etiqueta_provincia", "etiqueta_categoria"])["conteo"]
    .sum()
    .unstack("etiqueta_categoria")
    .fillna(0).astype(int)
)
sexo_prov.columns.name = None
sexo_prov["Total"] = sexo_prov.sum(axis=1)
sexo_prov["% Mujer"] = (sexo_prov[CAT_MUJER] / sexo_prov["Total"] * 100).round(1)
sexo_prov["% Varón"] = (sexo_prov[CAT_VARON] / sexo_prov["Total"] * 100).round(1)
sexo_prov = sexo_prov.sort_values("Total", ascending=False)
sexo_prov

In [ ]:
# Tabla nacional
sexo_nacional = tabla_n_pct(
    df_sexo.groupby("etiqueta_categoria")["conteo"].sum(), POB_TOTAL
)
print("=== NACIONAL ===")
print(sexo_nacional.to_string(index=False))

In [ ]:
# Gráfico: población por provincia abierta por género
plot_df = sexo_prov[[CAT_MUJER, CAT_VARON]].sort_values("Total".replace("Total", CAT_MUJER), ascending=True)
plot_df = sexo_prov[[CAT_MUJER, CAT_VARON]].sort_values(CAT_MUJER, ascending=True)

fig, ax = plt.subplots(figsize=(9, 9))
y = range(len(plot_df))
ax.barh(list(y), plot_df[CAT_MUJER], color=MUJER, alpha=0.85, label="Mujer")
ax.barh(list(y), plot_df[CAT_VARON], left=plot_df[CAT_MUJER], color=VARON, alpha=0.85, label="Varón")

for i, (prov, row) in enumerate(plot_df.iterrows()):
    total = row[CAT_MUJER] + row[CAT_VARON]
    ax.text(total + 10_000, i, f"{total/1e6:.2f}M", va="center", fontsize=8)

ax.set_yticks(list(y))
ax.set_yticklabels(plot_df.index)
ax.set_xlabel("Población")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.0f}M"))
ax.set_title("Población por provincia y sexo — Argentina (Censo 2022)")
ax.legend()
plt.tight_layout()
plt.show()

---
## 3. Pirámide poblacional

La pirámide muestra la distribución por grupos quinquenales de edad.
Como sexo y edad son variables independientes en la base, se presentan por separado.
La pirámide usa el total de cada grupo de edad.

In [ ]:
# Tabla quinquenal nacional
quin_nac = (
    df_edad
    .groupby(["valor_categoria", "etiqueta_categoria"])["conteo"]
    .sum().reset_index()
    .sort_values("valor_categoria")
    .rename(columns={"etiqueta_categoria": "grupo_edad", "conteo": "N"})
)
quin_nac["%"] = (quin_nac["N"] / quin_nac["N"].sum() * 100).round(2)
print("=== PIRÁMIDE NACIONAL ===")
print(quin_nac[["grupo_edad", "N", "%"]].to_string(index=False))

In [ ]:
# Gráfico pirámide nacional
def graficar_piramide(quin_df, titulo):
    fig, ax = plt.subplots(figsize=(7, 9))
    y = range(len(quin_df))
    ax.barh(list(y), quin_df["N"], color=TOTAL, alpha=0.8)
    for i, row in enumerate(quin_df.itertuples()):
        ax.text(row.N + 5_000, i, f"{row.N/1e3:.0f}k  ({row._3:.1f}%)",
                va="center", fontsize=8)
    ax.set_yticks(list(y))
    ax.set_yticklabels(quin_df["grupo_edad"])
    ax.set_xlabel("Personas")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
    ax.set_xlim(0, quin_df["N"].max() * 1.3)
    ax.set_title(titulo)
    plt.tight_layout()
    plt.show()

graficar_piramide(quin_nac, "Pirámide de edad — Argentina (Censo 2022)")

In [ ]:
# Pirámide por provincia — cambiá este valor
PROV_PIRAMIDE = "Córdoba"

df_edad_prov = ciut.query(variables="PERSONA_EDADQUI", provincia=PROV_PIRAMIDE)

quin_prov = (
    df_edad_prov
    .groupby(["valor_categoria", "etiqueta_categoria"])["conteo"]
    .sum().reset_index()
    .sort_values("valor_categoria")
    .rename(columns={"etiqueta_categoria": "grupo_edad", "conteo": "N"})
)
quin_prov["%"] = (quin_prov["N"] / quin_prov["N"].sum() * 100).round(2)

print(f"=== PIRÁMIDE: {PROV_PIRAMIDE} ===")
print(quin_prov[["grupo_edad", "N", "%"]].to_string(index=False))

In [ ]:
graficar_piramide(quin_prov, f"Pirámide de edad — {PROV_PIRAMIDE} (Censo 2022)")

In [ ]:
# Tabla comparativa de estructura etaria por provincia
# (grandes grupos: 0-14, 15-64, 65+)
df_edadgru = ciut.query(variables="PERSONA_EDADGRU")

edad_prov = (
    df_edadgru
    .groupby(["etiqueta_provincia", "valor_categoria", "etiqueta_categoria"])["conteo"]
    .sum().reset_index()
    .pivot_table(index="etiqueta_provincia", columns="etiqueta_categoria",
                 values="conteo", aggfunc="sum")
    .fillna(0).astype(int)
)
edad_prov.columns.name = None
edad_prov["Total"] = edad_prov.sum(axis=1)
# % por columna
for col in [c for c in edad_prov.columns if c != "Total"]:
    edad_prov[f"% {col[:6]}"] = (edad_prov[col] / edad_prov["Total"] * 100).round(1)
edad_prov.sort_values("Total", ascending=False)

---
## 4. Nivel educativo — nacional y por provincia

In [ ]:
# Categorías reales del nivel educativo
ciut.describe("PERSONA_MNI")

In [ ]:
# Tabla nacional de nivel educativo (excluye ignorado = código 99)
educ_nac = (
    df_educ[df_educ["valor_categoria"] != "99"]
    .groupby(["valor_categoria", "etiqueta_categoria"])["conteo"]
    .sum().reset_index()
    .sort_values("valor_categoria")
    .rename(columns={"etiqueta_categoria": "nivel", "conteo": "N"})
)
educ_nac["%"] = (educ_nac["N"] / educ_nac["N"].sum() * 100).round(1)
print("=== NIVEL EDUCATIVO NACIONAL ===")
print(educ_nac[["nivel", "N", "%"]].to_string(index=False))

In [ ]:
# Gráfico nacional
fig, ax = plt.subplots(figsize=(8, 5))
y = range(len(educ_nac))
ax.barh(list(y), educ_nac["N"], color=TOTAL, alpha=0.85)

for i, row in enumerate(educ_nac.itertuples()):
    ax.text(row.N + 10_000, i, f"{row.N/1e6:.2f}M  ({row._4}%)",
            va="center", fontsize=8)

ax.set_yticks(list(y))
ax.set_yticklabels(educ_nac["nivel"])
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.0f}M"))
ax.set_xlim(0, educ_nac["N"].max() * 1.35)
ax.set_title("Máximo nivel de instrucción — Argentina (Censo 2022)")
plt.tight_layout()
plt.show()

In [ ]:
# Tabla por provincia (% de cada nivel)
educ_prov = (
    df_educ[df_educ["valor_categoria"] != "99"]
    .groupby(["etiqueta_provincia", "etiqueta_categoria"])["conteo"]
    .sum()
    .unstack("etiqueta_categoria")
    .fillna(0).astype(int)
)
educ_prov.columns.name = None
educ_prov["Total"] = educ_prov.sum(axis=1)
educ_prov_pct = educ_prov.drop(columns="Total").div(educ_prov["Total"], axis=0).mul(100).round(1)
educ_prov_pct.sort_values(educ_prov_pct.columns[-1], ascending=False)

In [ ]:
# Nivel educativo por provincia — apertura por sexo (distribuciones paralelas)
PROV_EDUC = "Buenos Aires"  # Cambiá esta provincia

df_educ_p = ciut.query(variables="PERSONA_MNI",  provincia=PROV_EDUC)
df_sexo_p = ciut.query(variables="PERSONA_P02",  provincia=PROV_EDUC)

educ_p = (
    df_educ_p[df_educ_p["valor_categoria"] != "99"]
    .groupby(["valor_categoria", "etiqueta_categoria"])["conteo"]
    .sum().reset_index()
    .sort_values("valor_categoria")
    .rename(columns={"etiqueta_categoria": "nivel", "conteo": "N"})
)
educ_p["%"] = (educ_p["N"] / educ_p["N"].sum() * 100).round(1)

sexo_p = (
    df_sexo_p
    .groupby("etiqueta_categoria")["conteo"]
    .sum().reset_index()
    .rename(columns={"etiqueta_categoria": "sexo", "conteo": "N"})
)
sexo_p["%"] = (sexo_p["N"] / sexo_p["N"].sum() * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Educación
axes[0].barh(range(len(educ_p)), educ_p["N"], color=TOTAL, alpha=0.85)
axes[0].set_yticks(range(len(educ_p)))
axes[0].set_yticklabels(educ_p["nivel"], fontsize=8)
for i, row in enumerate(educ_p.itertuples()):
    axes[0].text(row.N + 500, i, f"{row._4}%", va="center", fontsize=8)
axes[0].set_title(f"Nivel educativo — {PROV_EDUC}")
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e3:.0f}k"))
axes[0].set_xlim(0, educ_p["N"].max() * 1.3)

# Sexo
axes[1].bar(sexo_p["sexo"], sexo_p["N"], color=[MUJER, VARON], alpha=0.85, width=0.4)
for i, row in enumerate(sexo_p.itertuples()):
    axes[1].text(i, row.N + 1000, f"{row.N/1e6:.2f}M\n({row._4}%)",
                 ha="center", fontsize=9)
axes[1].set_title(f"Distribución por sexo — {PROV_EDUC}")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
axes[1].set_ylim(0, sexo_p["N"].max() * 1.25)

plt.suptitle(f"{PROV_EDUC} — Censo 2022", fontsize=12)
plt.tight_layout()
plt.show()

---
## 5. Déficit habitacional — NBI Total

In [ ]:
ciut.describe("HOGAR_NBI_TOT")

In [ ]:
# NBI total nacional
nbi_nac = tabla_n_pct(
    df_nbi_tot.groupby("etiqueta_categoria")["conteo"].sum(), HOGAR_TOTAL
)
nbi_nac.columns = ["categoria", "N", "%"]
print("=== NBI TOTAL NACIONAL ===")
print(nbi_nac.to_string(index=False))

In [ ]:
# Tabla NBI por provincia (N y %)
nbi_prov = (
    df_nbi_tot
    .groupby(["etiqueta_provincia", "etiqueta_categoria"])["conteo"]
    .sum().unstack("etiqueta_categoria")
    .fillna(0).astype(int)
)
nbi_prov.columns.name = None
nbi_prov["Total hogares"] = nbi_prov.sum(axis=1)

# Columna con NBI (la que no dice "sin")
col_con_nbi = [c for c in nbi_prov.columns if "Sin" not in c and "Total" not in c][0]
col_sin_nbi = [c for c in nbi_prov.columns if "Sin" in c][0]

nbi_prov["% con NBI"] = (nbi_prov[col_con_nbi] / nbi_prov["Total hogares"] * 100).round(1)
nbi_prov = nbi_prov.sort_values("% con NBI", ascending=False)
nbi_prov

In [ ]:
# Gráfico NBI por provincia
plot_nbi = nbi_prov[["% con NBI"]].sort_values("% con NBI")

fig, ax = plt.subplots(figsize=(8, 8))
colores_nbi = [NARANJA if v > plot_nbi["% con NBI"].mean() else VERDE
               for v in plot_nbi["% con NBI"]]
ax.barh(plot_nbi.index, plot_nbi["% con NBI"], color=colores_nbi, alpha=0.85)
ax.axvline(plot_nbi["% con NBI"].mean(), color="grey", linestyle="--",
           linewidth=0.9, label=f"Promedio: {plot_nbi['% con NBI'].mean():.1f}%")

for prov, val in plot_nbi["% con NBI"].items():
    ax.text(val + 0.2, list(plot_nbi.index).index(prov), f"{val}%",
            va="center", fontsize=8)

ax.set_xlabel("% hogares con NBI")
ax.set_title("Hogares con NBI por provincia — Argentina (Censo 2022)")
ax.legend()
plt.tight_layout()
plt.show()

---
## 6. Componentes del NBI

In [ ]:
# Resumen de los 5 componentes — nacional
nbi_vars = {
    "NBI Escolaridad"    : df_nbi_esc,
    "NBI Hacinamiento"   : df_nbi_hac,
    "NBI Saneamiento"    : df_nbi_san,
    "NBI Subsistencia"   : df_nbi_sub,
    "NBI Vivienda"       : df_nbi_viv,
}

def pct_con_nbi(df):
    """Extrae el % de hogares con NBI de un df de variable binaria."""
    resumen = df.groupby("etiqueta_categoria")["conteo"].sum()
    total = resumen.sum()
    col_nbi = [c for c in resumen.index if "Sin" not in c][0]
    n_nbi   = resumen[col_nbi]
    return n_nbi, total, round(n_nbi / total * 100, 1)

filas = []
for nombre, df in nbi_vars.items():
    n, t, pct = pct_con_nbi(df)
    filas.append({"Componente": nombre, "N hogares con NBI": n,
                  "Total hogares": t, "% con NBI": pct})

resumen_nbi = pd.DataFrame(filas).sort_values("% con NBI", ascending=False)
print("=== COMPONENTES NBI NACIONAL ===")
print(resumen_nbi.to_string(index=False))

In [ ]:
# Gráfico componentes
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(resumen_nbi["Componente"], resumen_nbi["% con NBI"],
        color=NARANJA, alpha=0.85)

for i, row in enumerate(resumen_nbi.itertuples()):
    ax.text(row._4 + 0.1, i, f"{row._4}%  ({row._2/1e3:.0f}k hogares)",
            va="center", fontsize=9)

ax.set_xlabel("% hogares afectados")
ax.set_xlim(0, resumen_nbi["% con NBI"].max() * 1.5)
ax.set_title("Componentes NBI — Argentina (Censo 2022)")
plt.tight_layout()
plt.show()

In [ ]:
# Componentes NBI por provincia
def nbi_por_provincia(df):
    prov = (
        df.groupby(["etiqueta_provincia", "etiqueta_categoria"])["conteo"]
        .sum().unstack().fillna(0).astype(int)
    )
    prov.columns.name = None
    total = prov.sum(axis=1)
    col_nbi = [c for c in prov.columns if "Sin" not in c][0]
    return (prov[col_nbi] / total * 100).round(1)

tabla_comp = pd.DataFrame({
    nombre: nbi_por_provincia(df)
    for nombre, df in nbi_vars.items()
})
tabla_comp["NBI Total"] = nbi_prov["% con NBI"]
tabla_comp.sort_values("NBI Total", ascending=False)

---
## 7. Hacinamiento de hogares

In [ ]:
ciut.describe("HOGAR_H20CP")

In [ ]:
# Hacinamiento nacional
hacin_nac = (
    df_hacin
    .groupby(["valor_categoria", "etiqueta_categoria"])["conteo"]
    .sum().reset_index()
    .sort_values("valor_categoria")
    .rename(columns={"etiqueta_categoria": "categoria", "conteo": "N"})
)
hacin_nac["%"] = (hacin_nac["N"] / hacin_nac["N"].sum() * 100).round(1)
print("=== HACINAMIENTO NACIONAL ===")
print(hacin_nac[["categoria", "N", "%"]].to_string(index=False))

In [ ]:
# Gráfico hacinamiento
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(range(len(hacin_nac)), hacin_nac["N"],
        color=[VERDE if "Sin" in c else NARANJA for c in hacin_nac["categoria"]],
        alpha=0.85)
ax.set_yticks(range(len(hacin_nac)))
ax.set_yticklabels(hacin_nac["categoria"])
for i, row in enumerate(hacin_nac.itertuples()):
    ax.text(row.N + 500, i, f"{row.N/1e3:.0f}k  ({row._4}%)",
            va="center", fontsize=9)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
ax.set_xlim(0, hacin_nac["N"].max() * 1.35)
ax.set_title("Hacinamiento de hogares — Argentina (Censo 2022)")
plt.tight_layout()
plt.show()

In [ ]:
# Hacinamiento por provincia
hacin_prov = (
    df_hacin
    .groupby(["etiqueta_provincia", "etiqueta_categoria"])["conteo"]
    .sum().unstack().fillna(0).astype(int)
)
hacin_prov.columns.name = None
hacin_prov["Total"] = hacin_prov.sum(axis=1)
col_hac = [c for c in hacin_prov.columns if "Sin" not in c and "Total" not in c]
hacin_prov["% con hacinamiento"] = (
    hacin_prov[col_hac].sum(axis=1) / hacin_prov["Total"] * 100
).round(1)
hacin_prov.sort_values("% con hacinamiento", ascending=False)

---
## 8. Índice de Privación Material de los Hogares (IPMH)

In [ ]:
ciut.describe("HOGAR_IPMH")

In [ ]:
# IPMH nacional
ipmh_nac = (
    df_ipmh
    .groupby(["valor_categoria", "etiqueta_categoria"])["conteo"]
    .sum().reset_index()
    .sort_values("valor_categoria")
    .rename(columns={"etiqueta_categoria": "categoria", "conteo": "N"})
)
ipmh_nac["%"] = (ipmh_nac["N"] / ipmh_nac["N"].sum() * 100).round(1)
print("=== IPMH NACIONAL ===")
print(ipmh_nac[["categoria", "N", "%"]].to_string(index=False))

In [ ]:
# Gráfico IPMH
fig, ax = plt.subplots(figsize=(7, 4))
colores_ipmh = [VERDE, NARANJA, "#d94f4f", "#8B0000"]
ax.barh(range(len(ipmh_nac)), ipmh_nac["N"],
        color=colores_ipmh[:len(ipmh_nac)], alpha=0.85)
ax.set_yticks(range(len(ipmh_nac)))
ax.set_yticklabels(ipmh_nac["categoria"])
for i, row in enumerate(ipmh_nac.itertuples()):
    ax.text(row.N + 500, i, f"{row.N/1e6:.2f}M  ({row._4}%)",
            va="center", fontsize=9)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.0f}M"))
ax.set_xlim(0, ipmh_nac["N"].max() * 1.4)
ax.set_title("Índice de Privación Material de Hogares (IPMH)\nArgentina — Censo 2022")
plt.tight_layout()
plt.show()

In [ ]:
# IPMH por provincia
ipmh_prov = (
    df_ipmh
    .groupby(["etiqueta_provincia", "etiqueta_categoria"])["conteo"]
    .sum().unstack().fillna(0).astype(int)
)
ipmh_prov.columns.name = None
ipmh_prov["Total"] = ipmh_prov.sum(axis=1)
for col in [c for c in ipmh_prov.columns if c != "Total"]:
    ipmh_prov[f"% {col[:15]}"] = (ipmh_prov[col] / ipmh_prov["Total"] * 100).round(1)
ipmh_prov.sort_values("Total", ascending=False)

---
## 9. Resumen déficit habitacional por provincia

In [ ]:
# Tabla resumen de todos los indicadores de deficit
deficit = pd.DataFrame(index=nbi_prov.index)
deficit["% NBI Total"]       = nbi_prov["% con NBI"]
deficit["% NBI Escolaridad"] = nbi_por_provincia(df_nbi_esc)
deficit["% NBI Hacinamiento"]= nbi_por_provincia(df_nbi_hac)
deficit["% NBI Saneamiento"] = nbi_por_provincia(df_nbi_san)
deficit["% NBI Subsistencia"]= nbi_por_provincia(df_nbi_sub)
deficit["% NBI Vivienda"]    = nbi_por_provincia(df_nbi_viv)
deficit["% Hacinamiento"]    = hacin_prov["% con hacinamiento"]

deficit.sort_values("% NBI Total", ascending=False)

In [ ]:
# Heatmap de déficit por provincia
import matplotlib.colors as mcolors

fig, ax = plt.subplots(figsize=(12, 9))
data = deficit.sort_values("% NBI Total", ascending=False)

im = ax.imshow(data.values, cmap="YlOrRd", aspect="auto")
ax.set_xticks(range(len(data.columns)))
ax.set_xticklabels(data.columns, rotation=30, ha="right", fontsize=9)
ax.set_yticks(range(len(data.index)))
ax.set_yticklabels(data.index, fontsize=9)

for i in range(len(data.index)):
    for j in range(len(data.columns)):
        val = data.values[i, j]
        if not pd.isna(val):
            ax.text(j, i, f"{val:.1f}%", ha="center", va="center",
                    fontsize=7, color="black" if val < 15 else "white")

plt.colorbar(im, ax=ax, label="%")
ax.set_title("Indicadores de déficit habitacional por provincia — Censo 2022")
plt.tight_layout()
plt.show()

---
## 10. Exportar

In [ ]:
with pd.ExcelWriter("analisis_censal_2022.xlsx") as writer:
    sexo_prov.to_excel(writer, sheet_name="Poblacion_sexo")
    edad_prov.to_excel(writer, sheet_name="Grupos_edad")
    educ_prov.to_excel(writer, sheet_name="Nivel_educativo")
    nbi_prov.to_excel(writer, sheet_name="NBI_total")
    tabla_comp.to_excel(writer, sheet_name="NBI_componentes")
    hacin_prov.to_excel(writer, sheet_name="Hacinamiento")
    ipmh_prov.to_excel(writer, sheet_name="IPMH")
    deficit.to_excel(writer, sheet_name="Deficit_resumen")

print("Exportado: analisis_censal_2022.xlsx")